In [3]:
import torch
from vector_quantize_pytorch import VectorQuantize

vq = VectorQuantize(
    dim = 512,
    codebook_size = 1024,     # codebook size
    decay = 0.8,             # the exponential moving average decay, lower means the dictionary will change faster
    commitment_weight = 1.   # the weight on the commitment loss
)

x = torch.randn(1, 1024, 512)
quantized, indices, commit_loss = vq(x) # (1, 1024, 256), (1, 1024), (1)

quantized.shape, indices.shape, commit_loss.shape

(torch.Size([1, 1024, 512]), torch.Size([1, 1024]), torch.Size([1]))

In [ ]:
# vq_opt.num_quantizers = 8
#     vq_opt.codebook_size = 1024
#     vq_opt.code_dim = 512
#     vq_opt.vq_group = 2

In [6]:

import torch
from vector_quantize_pytorch import ResidualVQ

residual_vq = ResidualVQ(
    dim = 512,
    num_quantizers = 8,      # specify number of quantizers
    codebook_size = 1024,    # codebook size
)

x = torch.randn(32, 48, 512)

quantized, indices, commit_loss = residual_vq(x)
print(quantized.shape, indices.shape, commit_loss.shape)
# (1, 1024, 256), (1, 1024, 8), (1, 8)

# if you need all the codes across the quantization layers, just pass return_all_codes = True

quantized, indices, commit_loss, all_codes = residual_vq(x, return_all_codes = True)
print(quantized.shape, indices.shape, commit_loss.shape, all_codes.shape)


torch.Size([32, 48, 512]) torch.Size([32, 48, 8]) torch.Size([1, 8])
torch.Size([32, 48, 512]) torch.Size([32, 48, 8]) torch.Size([1, 8]) torch.Size([8, 32, 48, 512])


In [18]:
import torch
from vector_quantize_pytorch import GroupedResidualVQ

residual_vq = GroupedResidualVQ(
    dim = 512,
    num_quantizers = 8,      # specify number of quantizers
    groups = 2,
    codebook_size = 1024,    # codebook size
)

x = torch.randn(32, 48, 512)

quantized, indices, commit_loss = residual_vq(x)
print(quantized.shape, indices.shape, commit_loss.shape)

torch.Size([32, 48, 512]) torch.Size([2, 32, 48, 8]) torch.Size([2, 1, 8])


In [19]:
residual_vq.codebooks.shape

torch.Size([2, 8, 1024, 256])

In [15]:
a = residual_vq.get_codes_from_indices(indices)

a = a.permute(1,2,3,4,0)
a = a.reshape(a.shape[0], a.shape[1], a.shape[2], a.shape[3]*a.shape[4])
a.shape

torch.Size([8, 32, 48, 512])

In [17]:
b = residual_vq.get_output_from_indices(indices)
b.shape,indices.shape

(torch.Size([32, 48, 512]), torch.Size([2, 32, 48, 8]))

In [ ]:
#   @property
#     def codebooks(self):
#         return torch.stack(tuple(rvq.codebooks for rvq in self.rvqs))

#     @property
#     def split_dim(self):
#         return 1 if self.accept_image_fmap else -1

#     def get_codes_from_indices(self, indices):
#         codes = tuple(rvq.get_codes_from_indices(chunk_indices) for rvq, chunk_indices in zip(self.rvqs, indices))
#         return torch.stack(codes)

#     def get_output_from_indices(self, indices):
#         outputs = tuple(rvq.get_output_from_indices(chunk_indices) for rvq, chunk_indices in zip(self.rvqs, indices))
#         return torch.cat(outputs, dim = self.split_dim)


In [1]:
import torch
from vector_quantize_pytorch import GroupedResidualVQ

residual_vq = GroupedResidualVQ(
    dim = 512,
    num_quantizers = 8,      # specify number of quantizers
    groups = 2,
    codebook_size = 1024,    # codebook size
)

x = torch.randn(32, 48, 512)

quantized, indices, commit_loss = residual_vq(x)
print(quantized.shape, indices.shape, commit_loss.shape)

torch.Size([32, 48, 512]) torch.Size([2, 32, 48, 8]) torch.Size([2, 1, 8])


In [2]:
residual_vq.get_codes_from_indices(indices).shape

torch.Size([2, 8, 32, 48, 256])

In [11]:
myind = indices[0:1,:,:,0].unsqueeze(-1)
# Assuming my_tensor is your tensor
myindX = myind.repeat(1, 1, 1, 8)
myindX.shape

torch.Size([1, 32, 48, 8])

In [7]:
residual_vq.get_codes_from_indices(myind).shape

torch.Size([1, 8, 32, 48, 256])

In [22]:
myindi = torch.zeros((32,48),dtype=torch.int64).unsqueeze(0)
indi = torch.concatenate((myindi,myindi),dim=0)
indi.shape

#replicate indi 8 times
indif = indi.unsqueeze(3).expand(-1,-1,-1,8)
indif.shape

torch.Size([2, 32, 48, 8])

In [23]:
residual_vq.get_codes_from_indices(indif)

tensor([[[[[-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           ...,
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01]],

          [[-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
           [-1.4668e-01, -2.6855e-01,  5.0703e-01,  ...,  1.0013e+00,
             9.9408e-02,  2.5356e-01],
 

In [25]:
x = torch.zeros((32,48),dtype=torch.int64)
x1_ = x.unsqueeze(0)
x2_ = x.unsqueeze(0)
indi = torch.cat((x1_,x2_),dim=0)

indif = indi.unsqueeze(3).expand(-1,-1,-1,8)

huso = residual_vq.get_codes_from_indices(indif)#.shape
# x_d = self.quantizer.get_codes_from_indices(indif)
# # x_d = x_d.view(1, -1, self.code_dim).permute(0, 2, 1).contiguous()
# x = x_d.sum(dim=0).permute(0, 2, 1)

In [27]:
hso

NameError: name 'hso' is not defined